In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import os
import pandas as pd
import torch

from src.data import GraphTimeSeriesLoader, GNNTimeSeriesDataset

%load_ext autoreload
%autoreload 2

In [3]:
processed_data_dir = "data/datos_procesados"
dataset_filenames = {
    "buildings": f"{processed_data_dir}/buildings/roads_buildings.csv",
    "bus_stops": f"{processed_data_dir}/bus_stops/bus_stops_per_road.parquet",
    "events": f"{processed_data_dir}/events/events_dataset.csv",
    "incidents": f"{processed_data_dir}/incidents/incidents_dataset.csv",
    "irregularities": [f"{processed_data_dir}/irregularidades_procesadas/" + f for f in os.listdir(f"{processed_data_dir}/irregularidades_procesadas/") if f.endswith('.parquet')],
    # "coi_points": "pmts_processed/coi_puntos_dataset.parquet",
    # "coi_parks": "pmts_processed/coi_parques_dataset.parquet",
    "coi_points": [f"{processed_data_dir}/pmts_processed/coi_puntos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_puntos/") if f.endswith('.parquet')],
    "coi_tracts": [f"{processed_data_dir}/pmts_processed/coi_tramos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_tramos/") if f.endswith('.parquet')],
    "road_state": f"{processed_data_dir}/road_state/road_state_dateset.parquet", # TODO: Should be 'dataset' no 'dateset'
}
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"

In [4]:
loader = GraphTimeSeriesLoader(
    graph_pkl_filename=graph_filename,
    buildings_csv_filename=dataset_filenames["buildings"],
    bus_stops_parquet_filename=dataset_filenames["bus_stops"],
    events_csv_filename=dataset_filenames["events"],
    incidents_csv_filename=dataset_filenames["incidents"],
    road_state_parquet_filename=dataset_filenames["road_state"],
    irregularities_filenames=dataset_filenames["irregularities"],
    coi_points_filenames=dataset_filenames["coi_points"],
    coi_tracts_filenames=dataset_filenames["coi_tracts"],
)

In [5]:
# incident_set = set([tuple(l) for l in loader.incidents_df[["road_id", "timeframe"]].values])
# event_set = set([tuple(l) for l in loader.events_df[["road_id", "timeframe"]].values])
# common_tuples = incident_set.intersection(event_set)
# list(common_tuples)[:5]

In [6]:
# # Convert the nx graph to a pandas DataFrame to check that everything is loaded correctly
# all_nodes_df = pd.DataFrame.from_dict(dict(loader.base_graph.nodes(data=True)), orient='index')
# all_nodes_df

In [9]:
# Example usage of the loader to load data for specific nodes and time windows
nodes_to_load = ["9004322_TF.1", "50008506_TF.1", "12003262_FT.1"]
time_windows_to_load = [
    # Query for 2 hours (8 * 15 mins)
    ('2024-09-29 23:00:00', 8),
    # Query for 1 hour (4 * 15 mins)
    ('2024-08-24 15:30:00', 4),
    ('2024-02-24 01:30:00', 4)
]

# Load the data
static_features, temporal_features = loader.load_data(
    nodes_to_load, time_windows_to_load
)


--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (48, 42) ---


In [10]:
static_features

,id_mvi,MVICODIGO,MVICCALZAD,MVICIV,MVICCAT,MVITCLA,MVITIPO,MVINOMBRE,MVINALTERN,MVINPRINCI,...,warehouse,water_tank,water_tankmm,yes,yesmm,num_stops,num_stops_m,num_stops_s,mean_distance_meters,median_distance_meters
node_id,,,,,,,,,,,,,,,,,,,,,
9004322_TF.1,9004322_TF.1,116225,228531.0,9004322,9004150,1,AC,AVENIDA LUIS CARLOS GALAN,,0240000000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
50008506_TF.1,50008506_TF.1,51675,272273.0,50008506,12000648,1,AC,AVENIDA JOSE CELESTINO MUTIS,,0630000000,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
12003262_FT.1,12003262_FT.1,54323,902937.0,12003262,12001715,1,AK,AVENIDA CIUDAD DE QUITO,,0300000000,...,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Converting it to a PyTorch Dataset for GNNs
samples = [(node, '2024-01-01 00:00:00') for node in loader.base_graph.nodes()]
gnn_dataset = GNNTimeSeriesDataset(loader, loader.base_graph, samples)

# Then we can use it with a standard PyTorch DataLoader
# TODO: We may need to implement a custom collate_fn for batching graph data
torch_dataloader = torch.utils.data.DataLoader(
    gnn_dataset,
    batch_size=32,
    shuffle=True,
)

for batch in torch_dataloader:
    # Training loop, `batch` contains tensors ready for the model
    pass


--- Starting data load for 7 nodes ---
--- Load complete. Found 7 static records and 7 temporal records. ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records and 3 temporal records. ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records and 3 temporal records. ---

--- Starting data load for 5 nodes ---
--- Load complete. Found 5 static records and 5 temporal records. ---

--- Starting data load for 6 nodes ---
--- Load complete. Found 6 static records and 6 temporal records. ---

--- Starting data load for 4 nodes ---
--- Load complete. Found 4 static records and 4 temporal records. ---

--- Starting data load for 5 nodes ---


KeyboardInterrupt: 